In [1]:
import torch
import torch.nn.functional as F

import pandas as pd

from openai import OpenAI
import numpy as np

import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

EMBED_MODEL_SMALL = "text-embedding-3-small"  # 1536 вимірів
EMBED_MODEL_LARGE = "text-embedding-3-large"  # 3072 вимірів
LLM_MODEL = "gpt-4o-mini"  # для генерації відповідей
API_KEY = os.getenv("API_KEY")

client = OpenAI(api_key=API_KEY)

words = [
    "vision",
    "color",
    "red",
    "orange",
    "yellow",
    "green",
    "blue",
    "violet",
    "purple",
    "lilac",
    "taste",
    "bitter",
    "sweet",
    "sour"
]

def embed(text: str, model: str) -> np.ndarray:
    model_name = f"text-embedding-3-{model}"
    response = client.embeddings.create(model=model_name, input=text)
    return torch.tensor(response.data[0].embedding, dtype=torch.float32)
    # return np.array(response.data[0].embedding)

# cosine similarity
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [3]:
embeddings = {
    "small": torch.zeros(len(words), 1536),
    "large": torch.zeros(len(words), 3072),
}

for m in ["small", "large"]:
    for i in range(len(words)):
        embeddings[m][i] = embed(words[i], m)

In [69]:
# emb = {}
# for m in ["small", "large"]:
#     data = {}
#     for w in words:
#         data[w] = embed(w, m)
#     emb[m] = data

# # Sigmoid function
# thresholds = [0.05, 0.055, 0.065, 0.07]
# for m in ["small", "large"]:
#     for w, tensor in emb["large"].items():
#         # print(tensor)
#         for t in thresholds:
#             B = (tensor >= t).int()

#             print(m, "===", w, "===", t, "===", [item[0] for item in (B == 1).nonzero().tolist()])

In [ ]:
"""
получить координатьі пересечения тех елементов, которьіе ненулевьіе 
в матрице ковариаций. Например, если у слова red и white в пересечении стоит значение 2, 
то получить 2 значения индексов на которьіх значения ненулевьіе
"""

In [20]:
VERSION = "4"
PATH  = f"model_artifacts/version_{VERSION}/"
os.makedirs(PATH, exist_ok=True)

In [21]:
# Sigmoid

T_s = {
    "large": [0.055, 0.065], # [0.05, 0.055, 0.065, 0.07]
    "small": [0.07, 0.075, 0.08, 0.85] # [0.07, 0.075, 0.08, 0.085, 0.09]
}

try:
    for model in T_s:
        # data = []
        
        for t in T_s[model]:
            
            print(t, " -> ", model)
            
            # filtering
            B = (embeddings[model] >= t).int()
    
            C = B @ B.T

            model_path = PATH + f"{model}/"
            os.makedirs(model_path, exist_ok=True)

            # ### Integral matrices
            # ###
            # data.append(C.tolist())
            # result = [
            #     [list(values) for values in zip(*rows)]
            #     for rows in zip(*data)
            # ]
            # d_values = f"{T_s[MODEL][0]}-{T_s[MODEL][-1]}" 
            # ###

            
            df = pd.DataFrame(C, index=words, columns=words)
            df.to_csv(f"{model_path}/filtration_s_{t}.csv", mode="w")
            
except Exception as e:
    print(f"An error occurred: {e}")

0.055  ->  large
0.065  ->  large
0.07  ->  small
0.075  ->  small
0.08  ->  small
0.85  ->  small


In [22]:
# Gauss
T_g = {
    "large": [0.0001, 0.0003], # [0.05, 0.055, 0.065, 0.07]
    "small": [0.0007, 0.001] # [0.07, 0.075, 0.08, 0.085, 0.09]
}

try:
    # data = []
    for model in T_g:
        for t in T_g[model]:
            print(t, " -> ", model)
            
            # filtering
            B = (torch.abs(embeddings[model]) <= t).int()
    
            C = B @ B.T

            model_path = PATH + f"{model}/"
            os.makedirs(model_path, exist_ok=True)
        
            df = pd.DataFrame(C, index=words, columns=words)
            df.to_csv(f"{model_path}/filtration_g_{t}.csv", mode="w")
            
except Exception as e:
    print(f"An error occurred: {e}")

0.0001  ->  large
0.0003  ->  large
0.0007  ->  small
0.001  ->  small


In [25]:
B = (embeddings["large"] >= t).int()
thresholds = [0.05, 0.055, 0.065, 0.07]

for i in range(len(words)):
    idx = [x[0] for x in (B[i] == 1).nonzero().tolist()]
    
    print(words[i], " ==> ", idx)

vision  ==>  [31, 94, 1419]
color  ==>  [449, 1019, 1582, 2782]
red  ==>  [1019, 2782]
orange  ==>  [57, 261, 449, 1019, 2782]
yellow  ==>  [1019]
green  ==>  [57, 2782]
blue  ==>  [1019, 2782]
violet  ==>  [1019]
purple  ==>  [1019, 2782]
lilac  ==>  [57, 80, 1019]
taste  ==>  [31, 437, 771]
bitter  ==>  [1019]
sweet  ==>  [373, 1019]
sour  ==>  [30, 528, 1019, 2782]


In [18]:
# Gauss
deltas = {
    "small": [0.0007, 0.0008, 0.00085, 0.0009, 0.00095],
    "large": [0.0003, 0.0005, 0.0007, 0.00085]
}
# print(deltas)

try:
    for m in embeddings:
        for d in deltas[m]:
            # filtering
            B = (torch.abs(embeddings[m]) <= d).int()
            C = B @ B.T
        
            print(f"model_size: {m} -- delta = {d}, nonzoer(B) = {len((B == 1).nonzero())}")
        
            df = pd.DataFrame(C, index=words, columns=words)
            df.to_csv(f"model_artifacts/{iteration}/{m}/filtration_2-cov_matrix_{d}.csv", mode='x')

except Exception as e:
    # Catch specific exceptions and access the exception object
    print(f"An error occurred: {e}")

model_size: small -- delta = 0.0007, nonzoer(B) = 490
model_size: small -- delta = 0.0008, nonzoer(B) = 549
model_size: small -- delta = 0.00085, nonzoer(B) = 580
model_size: small -- delta = 0.0009, nonzoer(B) = 619
model_size: small -- delta = 0.00095, nonzoer(B) = 661
model_size: large -- delta = 0.0003, nonzoer(B) = 678
model_size: large -- delta = 0.0005, nonzoer(B) = 1097
model_size: large -- delta = 0.0007, nonzoer(B) = 1504
model_size: large -- delta = 0.00085, nonzoer(B) = 1814
